In [0]:
%run ../src/Bronze/landing_ecomm

In [0]:
%run ../src/Silver/Unified_ecomm

In [0]:
%run ../src/Gold/publish_ecomm

In [0]:
    import pytest
    import pyspark.sql.functions as F
    from pyspark.sql.window import Window
    from pyspark.sql.types import *

In [0]:
#testCase1
import pytest
import pyspark.sql.functions as F

def test_clean_column_spaces_removes_spaces():
    """
    Validates that clean_column_spaces strips all whitespace characters
    out of DataFrame header strings entirely.
    """
    mock_data = [("123", "Jatin", "Bangalore")]
    mock_columns = ["Customer ID", "Customer Name ", "Home Base"]
    
    input_df = spark.createDataFrame(mock_data, schema=mock_columns)
    processed_df = clean_column_spaces(input_df)
    expected_columns = ["CustomerID", "CustomerName", "HomeBase"]
    
    assert processed_df.columns == expected_columns

In [0]:
test_clean_column_spaces_removes_spaces()

In [0]:
customer_schema=StructType([
        StructField("CustomerID", StringType(), True),
        StructField("CustomerName", StringType(), True),
        StructField("email", StringType(), True),
        StructField("Segment", StringType(), True),
        StructField("Country", StringType(), True),
        StructField("PostalCode", StringType(), True),
        StructField("phone", StringType(), True)
    ])

In [0]:
#testCase2

def test_dqc_customer_pristine_record_accepted():
    """
    Ensures a perfectly valid customer row passes all checks,
    landing exclusively in accepted_df with no errors.
    """

    mock_data = [
        ("CUST001", "Jatin Sharma", "jatin@example.com", "Consumer", "India", "560001", "9876543210")
    ]
    input_df = spark.createDataFrame(mock_data, schema=customer_schema)
    input_df.createOrReplaceTempView("tmp_mock_customer_good")
    accepted_df, rejected_df = dqc_checks_customer("tmp_mock_customer_good")

    assert accepted_df.count() == 1
    assert rejected_df.count() == 0
    
    assert "dq_errors" not in accepted_df.columns

In [0]:
test_dqc_customer_pristine_record_accepted()

In [0]:
#testCase3

def test_dqc_customer_invalid_records_rejected(customer_schema):
    """
    Verifies that rows containing anomalies are routed to rejected_df 
    """
    dirty_data = [
        (None, "Sanjana", "sanjana@test.com", "InvalidSegment", "India", None, "1234567890"),
        ("CUST002", "Sehaj123", "sehaj@test.com", "Corporate", "India", "560002", "1234567890"),
        ("CUST003", "Amit", "amit@test.co", "Home Office", "India", "560003", "#REF! Error")
    ]
    input_df = spark.createDataFrame(dirty_data, schema=customer_schema)
    input_df.createOrReplaceTempView("tmp_mock_customer_dirty")

    accepted_df, rejected_df = dqc_checks_customer("tmp_mock_customer_dirty")

    # Assert: Verify routing isolation split maps out cleanly
    assert accepted_df.count() == 0
    assert rejected_df.count() == 3

    rejected_records = rejected_df.collect()

    # Inspect Row A Flags
    row_a_errors = rejected_records[0]["dq_errors"]
    assert "MISSING_CUSTOMER_ID" in row_a_errors
    assert "INVALID_SEGMENT" in row_a_errors
    assert "MISSING_POSTAL_CODE" in row_a_errors

    # Inspect Row B Flags
    row_b_errors = rejected_records[1]["dq_errors"]
    assert "NAME_CONTAINS_NUMBERS" in row_b_errors

    # Inspect Row C Flags
    row_c_errors = rejected_records[2]["dq_errors"]
    assert "PHONE_HAS_SYSTEM_ERROR" in row_c_errors
    assert "PHONE_INVALID_DIGIT_COUNT" in row_c_errors

In [0]:
test_dqc_customer_invalid_records_rejected(customer_schema)

In [0]:
#testCase4

def test_dqc_customer_duplicate_ids_rejected(customer_schema):
    """
    Tests that the Window.partitionBy function correctly catches and 
    flags multiple non-unique duplicate CustomerIDs.
    """
    duplicate_data = [
        ("CUST_DUP", "Alex", "alex1@test.com", "Consumer", "US", "10001", "1234567890"),
        ("CUST_DUP", "Alex", "alex2@test.com", "Consumer", "US", "10001", "0987654321")
    ]
    input_df = spark.createDataFrame(duplicate_data, schema=customer_schema)
    input_df.createOrReplaceTempView("tmp_mock_customer_dups")

    _, rejected_df = dqc_checks_customer("tmp_mock_customer_dups")

    assert rejected_df.count() == 2
    
    for row in rejected_df.collect():
        assert "DUPLICATE_CUSTOMER_ID" in row["dq_errors"]

In [0]:
test_dqc_customer_duplicate_ids_rejected(customer_schema)

In [0]:
order_schema=StructType([
        StructField("RowID", IntegerType(), True),
        StructField("OrderID", StringType(), True),
        StructField("CustomerID", StringType(), True),
        StructField("ProductID", StringType(), True),
        StructField("OrderDate", StringType(), True),
        StructField("ShipDate", StringType(), True),
        StructField("Quantity", IntegerType(), True),
        StructField("Price", DoubleType(), True),
        StructField("Discount", DoubleType(), True),
        StructField("Profit", DoubleType(), True),
        StructField("ShipMode", StringType(), True)
    ])

In [0]:
#testcase 5
def test_dqc_orders_pristine_record_accepted(order_schema):
    """
    Ensures that a flawless row with correct string-date formats ('d/M/yyyy')
    is accepted cleanly and splits entirely into the accepted DataFrame.
    """
    good_data = [
        (1, "ORD-2014-001", "CUST-101", "PROD-OFF-01", "30/4/2014", "2/5/2014", 3, 150.00, 0.0, 45.00, "Second Class")
    ]
    input_df = spark.createDataFrame(good_data, schema=order_schema)
    input_df.createOrReplaceTempView("tmp_mock_orders_clean")

    accepted_df, rejected_df = dqc_checks_orders("tmp_mock_orders_clean")

    assert accepted_df.count() == 1
    assert rejected_df.count() == 0

    assert "dq_errors" not in accepted_df.columns

In [0]:
test_dqc_orders_pristine_record_accepted(order_schema)

In [0]:
#testCase 6
def test_dqc_orders_date_anomalies_rejected(order_schema):
    """
    Validates both sides of date-safety checking:
    Row A: Identifies a logical error when OrderDate occurs after ShipDate.
    Row B: Identifies a malformed structure when date format string parsing fails.
    """
    faulty_date_data = [
        (2, "ORD-002", "CUST-102", "PROD-TEC-02", "15/5/2014", "2/5/2014", 1, 10.00, 0.0, 2.00, "Standard Class"),
        (3, "ORD-003", "CUST-103", "PROD-TEC-03", "2014-04-30", "2/5/2014", 1, 20.00, 0.0, 5.00, "Standard Class")
    ]
    input_df = spark.createDataFrame(faulty_date_data, schema=order_schema)
    input_df.createOrReplaceTempView("tmp_mock_orders_bad_dates")

    accepted_df, rejected_df = dqc_checks_orders("tmp_mock_orders_bad_dates")

    assert accepted_df.count() == 0
    assert rejected_df.count() == 2

    rejected_rows = rejected_df.orderBy("RowID").collect()

    assert "LOGICAL_ERROR_ORDER_DATE_AFTER_SHIP_DATE" in rejected_rows[0]["dq_errors"]

    assert "MALFORMED_ORDER_DATE_FORMAT" in rejected_rows[1]["dq_errors"]

In [0]:
test_dqc_orders_date_anomalies_rejected(order_schema)

In [0]:
#testCase 7
def test_dqc_orders_duplicate_row_ids_rejected(order_schema):
    """
    Ensures that Window.partitionBy detects matching RowID records 
    and applies a 'DUPLICATE_ROW_ID' flag across both conflicting instances.
    """

    duplicate_rows = [
        (999, "ORD-004", "CUST-104", "PROD-FUR-01", "01/5/2014", "05/5/2014", 2, 500.00, 0.1, 50.00, "First Class"),
        (999, "ORD-005", "CUST-104", "PROD-FUR-01", "01/5/2014", "05/5/2014", 2, 500.00, 0.1, 50.00, "First Class")
    ]
    input_df = spark.createDataFrame(duplicate_rows, schema=order_schema)
    input_df.createOrReplaceTempView("tmp_mock_orders_dups")

    accepted_df, rejected_df = dqc_checks_orders("tmp_mock_orders_dups")

    assert accepted_df.count() == 0
    assert rejected_df.count() == 2

    for row in rejected_df.collect():
        assert "DUPLICATE_ROW_ID" in row["dq_errors"]

In [0]:
test_dqc_orders_duplicate_row_ids_rejected(order_schema)

In [0]:
product_schema=StructType([
        StructField("ProductID", StringType(), True),
        StructField("Category", StringType(), True),
        StructField("Sub-Category", StringType(), True), # Intentionally left dirty with hyphen
        StructField("ProductName", StringType(), True),
        StructField("State", StringType(), True),
        StructField("Priceperproduct", StringType(), True) # Kept as string to test regex parsing
    ])

In [0]:
#testCase 8
def test_dqc_product_clean_record_accepted(product_schema):
    """
    Ensures that a valid record has its columns stripped of spaces/hyphens
    ('Sub-Category' -> 'SubCategory'), passes validations, and lands in accepted_df.
    """
    good_data = [
        ("PROD123", "Technology", "Phones", "iPhone 15", "Karnataka", "79999.00")
    ]
    input_df = spark.createDataFrame(good_data, schema=product_schema)
    input_df.createOrReplaceTempView("tmp_mock_products_clean")

    accepted_df, rejected_df = dqc_checks_product("tmp_mock_products_clean")

    assert accepted_df.count() == 1
    assert rejected_df.count() == 0
    
    assert "SubCategory" in accepted_df.columns
    assert "Sub-Category" not in accepted_df.columns
    

In [0]:
test_dqc_product_clean_record_accepted(product_schema)

In [0]:
#testCase9
def test_dqc_product_invalid_records_rejected(product_schema):
    """
    Validates that text data anomalies (like numbers in State names) are identified,
    routed to rejected_df, and combined into a clean comma-separated string column.
    """
    dirty_data = [
        ("PROD456", "Furniture", "Chairs", "Office Chair", "Karnat4k4!", "1500.00")
    ]
    input_df = spark.createDataFrame(dirty_data, schema=product_schema)
    input_df.createOrReplaceTempView("tmp_mock_products_dirty")

    accepted_df, rejected_df = dqc_checks_product("tmp_mock_products_dirty")

    assert accepted_df.count() == 0
    assert rejected_df.count() == 1

    rejected_row = rejected_df.first()
    expected_error_string = "STATE_CONTAINS_NUMBERS,STATE_CONTAINS_SPECIAL_CHARACTERS"
    assert rejected_row["dq_errors"] == expected_error_string

In [0]:
test_dqc_product_invalid_records_rejected(product_schema)

In [0]:
#testCase10
def test_dqc_product_invalid_prices_are_silently_dropped(product_schema):
    """
    Verifies the hard-coded top filter boundary: rows with alpha-characters 
    or negative flags in Priceperproduct are eliminated immediately from processing.
    """
    malformed_prices = [
        ("PROD789", "Office Supplies", "Art", "Markers", "Delhi", "FREE_PRICE"),
        ("PROD999", "Office Supplies", "Art", "Pens", "Delhi", "-25.00")
    ]
    input_df = spark.createDataFrame(malformed_prices, schema=product_schema)
    input_df.createOrReplaceTempView("tmp_mock_products_dropped")

    accepted_df, rejected_df = dqc_checks_product("tmp_mock_products_dropped")

    assert accepted_df.count() == 0
    assert rejected_df.count() == 0

In [0]:
test_dqc_product_invalid_prices_are_silently_dropped(product_schema)

In [0]:
#testCase11
def test_create_customer_product_bridge_execution():
    """
    Validates that create_customer_product deduplicates historical transactions,
    removes the customer 'state' field, and executes clean inner joins.
    """
    order_schema = StructType([
        StructField("CustomerID", StringType(), True),
        StructField("ProductID", StringType(), True),
        StructField("Quantity", IntegerType(), True)
    ])
    mock_orders = [
        ("CUST01", "PROD01", 2),
        ("CUST01", "PROD01", 5), 
        ("CUST02", "PROD02", 1)
    ]
    
    cust_schema = StructType([
        StructField("CustomerID", StringType(), True),
        StructField("CustomerName", StringType(), True),
        StructField("state", StringType(), True) # Target for dropping
    ])
    mock_customers = [
        ("CUST01", "Jatin Sharma", "Karnataka"),
        ("CUST02", "Sanjana", "Haryana")
    ]
    
    prod_schema = StructType([
        StructField("ProductID", StringType(), True),
        StructField("Category", StringType(), True)
    ])
    mock_products = [
        ("PROD01", "Technology"),
        ("PROD02", "Furniture")
    ]

    spark.createDataFrame(mock_orders, schema=order_schema).createOrReplaceTempView("v_test_orders")
    spark.createDataFrame(mock_customers, schema=cust_schema).createOrReplaceTempView("v_test_customers")
    spark.createDataFrame(mock_products, schema=prod_schema).createOrReplaceTempView("v_test_products")

    enriched_df = create_customer_product("v_test_orders", "v_test_customers", "v_test_products")


    assert enriched_df.count() == 2

    columns = enriched_df.columns
    assert "state" not in columns, "The customer 'state' column was not dropped!"
    assert "CustomerName" in columns
    assert "Category" in columns

    cust_1_record = enriched_df.filter(enriched_df.CustomerID == "CUST01").first()
    assert cust_1_record["CustomerName"] == "Jatin Sharma"
    assert cust_1_record["Category"] == "Technology"

In [0]:
test_create_customer_product_bridge_execution()

In [0]:
#testCase 12
def test_create_enriched_orders_left_joins_and_drops_state():
    """
    Validates that create_enriched_orders enriches transactions with customer/product data,
    retains unmatched transactions using left joins, and drops the customer 'state' field.
    """
    order_schema = StructType([
        StructField("OrderID", StringType(), True),
        StructField("CustomerID", StringType(), True),
        StructField("ProductID", StringType(), True),
        StructField("Profit", DoubleType(), True)
    ])
    mock_orders = [
        ("ORD_01", "CUST_01", "PROD_01", 120.50),
        ("ORD_02", "CUST_02", "MISSING_PROD", 45.00) 
    ]
    
    cust_schema = StructType([
        StructField("CustomerID", StringType(), True),
        StructField("CustomerName", StringType(), True),
        StructField("state", StringType(), True)  # Targeted for dropping
    ])
    mock_customers = [
        ("CUST_01", "Jatin Sharma", "Karnataka"),
        ("CUST_02", "Sanjana", "Haryana")
    ]
    
    prod_schema = StructType([
        StructField("ProductID", StringType(), True),
        StructField("Category", StringType(), True)
    ])
    mock_products = [
        ("PROD_01", "Technology")
    ]

    spark.createDataFrame(mock_orders, schema=order_schema).createOrReplaceTempView("v_orders_raw")
    spark.createDataFrame(mock_customers, schema=cust_schema).createOrReplaceTempView("v_customers_raw")
    spark.createDataFrame(mock_products, schema=prod_schema).createOrReplaceTempView("v_products_raw")

    enriched_df = create_enriched_orders("v_orders_raw", "v_customers_raw", "v_products_raw")

    assert enriched_df.count() == 2

    output_columns = enriched_df.columns
    assert "state" not in output_columns
    assert "CustomerName" in output_columns
    assert "Category" in output_columns

    matched_row = enriched_df.filter(F.col("OrderID") == "ORD_01").first()
    assert matched_row["CustomerName"] == "Jatin Sharma"
    assert matched_row["Category"] == "Technology"

    unmatched_row = enriched_df.filter(F.col("OrderID") == "ORD_02").first()
    assert unmatched_row["CustomerName"] == "Sanjana"
    assert unmatched_row["Category"] is None, "The Category field should fall back to None for unmatched left-join fields."

In [0]:
test_create_enriched_orders_left_joins_and_drops_state()

In [0]:
#testCase 13
def test_create_enriched_sales_transforms_and_selects_columns():
    """
    Validates that create_enriched_sales performs correct left joins,
    rounds profit metrics to 2 decimal places, and enforces strict column sequencing.
    """
    order_schema = StructType([
        StructField("RowID", IntegerType(), True),
        StructField("OrderID", StringType(), True),
        StructField("CustomerID", StringType(), True),
        StructField("ProductID", StringType(), True),
        StructField("OrderDate", StringType(), True),
        StructField("ShipDate", StringType(), True),
        StructField("Quantity", IntegerType(), True),
        StructField("Price", DoubleType(), True),
        StructField("Discount", DoubleType(), True),
        StructField("Profit", DoubleType(), True), # Testing rounding constraint
        StructField("ShipMode", StringType(), True)
    ])

    mock_orders = [
        (1, "ORD_01", "C_01", "P_01", "30/4/2014", "2/5/2014", 3, 100.0, 0.0, 45.126, "Standard Class"),
        (2, "ORD_02", "C_02", "MISSING_P", "01/5/2014", "05/5/2014", 1, 10.0, 0.0, -5.000, "Same Day")
    ]
    
    cust_schema = StructType([
        StructField("CustomerID", StringType(), True),
        StructField("CustomerName", StringType(), True),
        StructField("Country", StringType(), True)
    ])
    mock_customers = [
        ("C_01", "Jatin Sharma", "India"),
        ("C_02", "Sanjana", "India")
    ]
    
    prod_schema = StructType([
        StructField("ProductID", StringType(), True),
        StructField("Category", StringType(), True),
        StructField("SubCategory", StringType(), True)
    ])
    mock_products = [
        ("P_01", "Technology", "Phones")
    ]

    spark.createDataFrame(mock_orders, schema=order_schema).createOrReplaceTempView("v_silver_orders")
    spark.createDataFrame(mock_customers, schema=cust_schema).createOrReplaceTempView("v_silver_customers")
    spark.createDataFrame(mock_products, schema=prod_schema).createOrReplaceTempView("v_silver_products")


    result_df = create_enriched_sales(
        order_table="v_silver_orders", 
        cust_table="v_silver_customers", 
        prod_table="v_silver_products",
        target_table="dummy_gold_path"
    )


    assert result_df.count() == 2


    records = result_df.orderBy("RowID").collect()
    assert records[0]["Profit"] == 45.13, "Floating-point profit did not round up to 2 decimals properly!"
    assert records[1]["Profit"] == -5.00


    assert records[0]["Category"] == "Technology"
    assert records[1]["Category"] is None


    expected_column_sequence = [
        "OrderID", "OrderDate", "RowID", "Quantity", "Price", "Discount", 
        "ShipDate", "ShipMode", "Profit", "CustomerName", "Country", 
        "Category", "SubCategory", "CustomerID", "ProductID"
    ]
    assert result_df.columns == expected_column_sequence, "The output column sequence or names do not match the BI spec!"

In [0]:
test_create_enriched_sales_transforms_and_selects_columns()

In [0]:
#testCase 14

def test_create_gold_profit_aggregation_calculates_and_sorts_metrics():
    """
    Validates that create_gold_profit_aggregation extracts the correct year from strings,
    aggregates sums and rounds profits across group dimensions, and applies sorting rules.
    """

    enriched_schema = StructType([
        StructField("OrderDate", StringType(), True),
        StructField("Category", StringType(), True),
        StructField("SubCategory", StringType(), True),
        StructField("CustomerID", StringType(), True),
        StructField("CustomerName", StringType(), True),
        StructField("Profit", DoubleType(), True)
    ])


    mock_data = [
        ("21/8/2016", "Technology", "Phones", "C_01", "Jatin Sharma", 100.50),
        ("25/8/2016", "Technology", "Phones", "C_01", "Jatin Sharma", 50.05),
        ("02/11/2017", "Technology", "Phones", "C_01", "Jatin Sharma", 200.00),
        ("15/9/2016", "Technology", "Phones", "C_02", "Sanjana", 300.00)
    ]

    input_table_name = "tmp_mock_gold_enriched"
    spark.createDataFrame(mock_data, schema=enriched_schema).createOrReplaceTempView(input_table_name)

    aggregated_df = create_gold_profit_aggregation(input_table_name)


    results = aggregated_df.collect()
    assert len(results) == 3

    expected_columns = ["Year", "Category", "SubCategory", "CustomerID", "CustomerName", "TotalProfit"]
    assert aggregated_df.columns == expected_columns

    assert results[0]["Year"] == 2016
    assert results[0]["CustomerName"] == "Sanjana"
    assert results[0]["TotalProfit"] == 300.00

 
    assert results[1]["Year"] == 2016
    assert results[1]["CustomerName"] == "Jatin Sharma"
    assert results[1]["TotalProfit"] == 150.55  # 100.50 + 50.05

    assert results[2]["Year"] == 2017
    assert results[2]["TotalProfit"] == 200.00

In [0]:
test_create_gold_profit_aggregation_calculates_and_sorts_metrics()